In [1]:
import pandas as pd
import numpy as np
import networkx as nx

import os
from sklearn.metrics import accuracy_score, confusion_matrix
from scipy.stats import spearmanr, pearsonr, ttest_ind, percentileofscore
import itertools

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
AA_LIST = list("ACDEFGHIKLMNPQRSTVWYO-")
N_AA = len(AA_LIST)
aa_to_idx = {aa: i for i, aa in enumerate(AA_LIST)}
idx_to_aa = {v: k for k, v in aa_to_idx.items()}

def min_max_norm(data):
    data_min = np.min(data)
    data_max = np.max(data)
    normalized_data = (data - data_min) / (data_max - data_min)
    return normalized_data

def magnitude_norm(data, data_min=0):
    data_max = np.max(np.abs(data))
    normalized_data = (data - data_min) / (data_max - data_min)
    return normalized_data

def get_significance_mark(p_value, alpha=0.05):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < alpha:
        return '*'
    else:
        return 'n.s.'

In [3]:
selected_models = ["rep_5", "rep_6", "rep_7", "rep_9", "rep_10"]
root_dir = "/scratch/alpine/fana6609/ML/PLM-Epistasis"
attr_dir = "results/attribution_maps"
#attr_class = "class_1"
attn_dir = "results/attention_maps"
data_dir = "data"
task_type = "classification"

info_df = pd.read_csv(os.path.join(root_dir, data_dir, "input_info_VRC01_IC80.csv"))
aligned_sequences = np.array([list(seq) for seq in info_df['Sequence']])
seq_no_array = info_df['Seq_no'].values
true_labels = info_df["Label"].values
print(aligned_sequences.shape)
print(np.unique(true_labels, return_counts=True))

resno_df = pd.read_csv(os.path.join(root_dir, data_dir, "selected_residues_IC80.csv"))
resno_array = resno_df["ResLabel"].values

N_MODEL = len(selected_models)
N_SEQ, L_SEQ = aligned_sequences.shape

all_prob_0 = np.zeros([N_MODEL, N_SEQ])
all_prob_1 = np.zeros([N_MODEL, N_SEQ])

for n, model_id in enumerate(selected_models):
    predict_df = pd.read_csv(os.path.join(root_dir, "results/predictions",task_type,"train_"+model_id+".csv"))
    for s, pred in enumerate(predict_df["Prediction"]):
        pred_0 = float(pred.split()[1].split(",")[0])
        pred_1 = float(pred.split()[3].split("}")[0])
        assert predict_df["Seq_no"].iloc[s] == info_df["Seq_no"].iloc[s]
        all_prob_0[n, s] = pred_0
        all_prob_1[n, s] = pred_1

mean_prob_0 = np.mean(all_prob_0, axis=0)
mean_prob_1 = np.mean(all_prob_1, axis=0)

pred_labels = np.zeros(N_SEQ, dtype=int)
pred_labels[mean_prob_1 > mean_prob_0] = 1

data_df = info_df.loc[info_df['Seq_no'].isin(seq_no_array[np.where(true_labels == pred_labels)])]
aligned_sequences = np.array([list(seq) for seq in data_df['Sequence']])
class_label = data_df["Label"].values
seq_no_array = data_df['Seq_no'].values
print("--- After removing misprediction ---")
print(aligned_sequences.shape)
print(np.unique(class_label, return_counts=True))

(889, 209)
(array([0, 1]), array([583, 306]))
--- After removing misprediction ---
(856, 209)
(array([0, 1]), array([569, 287]))


In [4]:
N_MODEL = len(selected_models)
N_SEQ, L_SEQ = aligned_sequences.shape

all_attr_array_0 = np.zeros([N_MODEL, N_SEQ, L_SEQ])
all_attr_array_1 = np.zeros([N_MODEL, N_SEQ, L_SEQ])
attn_scalar_array = np.zeros((N_MODEL, N_SEQ, L_SEQ, L_SEQ))
cls_attn_scalar_array =  np.zeros([N_MODEL, N_SEQ, L_SEQ])

for n, model_name in enumerate(selected_models):
    for i, no in enumerate(data_df['Seq_no']):
        attr_0 = np.load(os.path.join(root_dir, attr_dir, task_type, model_name, str(no)+"_attribution_ig_class_0.npy"))
        all_attr_array_0[n,i] = magnitude_norm(attr_0)

        attr_1 = np.load(os.path.join(root_dir, attr_dir, task_type, model_name, str(no)+"_attribution_ig_class_1.npy"))
        all_attr_array_1[n,i] = magnitude_norm(attr_1)
        
        attn = np.load(os.path.join(root_dir, attn_dir, task_type, "full", model_name, str(no)+"_attentions.npy"))
        attn_arr = attn[1:, 1:]
        attn_scalar_array[n, i] = min_max_norm(attn_arr)
        cls_attn = attn[0, 1:]
        cls_attn_scalar_array[n,i] = min_max_norm(cls_attn)

In [5]:
all_correlations = np.empty((N_MODEL, N_SEQ, L_SEQ, L_SEQ))
for n in range(N_MODEL):
    for s in range(N_SEQ):
        data_slice = attn_scalar_array[n, s]
        corr_matrix = np.corrcoef(data_slice)
        all_correlations[n, s] = corr_matrix
print("Mean pairwise pearson correlation of residue-residue attention score")
print(f"mean={np.mean(all_correlations):.6f}, std={np.std(all_correlations):.4f}")
print("-> All residues in a sequence give attention to the same set of sequence in the same degree")

attn_score_array = np.mean(attn_scalar_array, axis=2)
arr_dict = {
    "[CLS]-to-Residue Attention" : cls_attn_scalar_array,
    "Residue-Residue Attention": attn_score_array,
}

for name, arr in arr_dict.items():
    print(f"\n{name}")
    reshaped_array = arr.reshape(N_MODEL, -1)
    corr, pvalue = spearmanr(reshaped_array, axis=1)
    #corr = np.corrcoef(reshaped_array)
    print("Correlation Matrix:\n", np.round(corr, 2))
    print("Mean off-diag corr:", np.mean(corr[np.triu_indices_from(corr, k=1)]).round(2))

cls_array = arr_dict["[CLS]-to-Residue Attention"].copy()
attn_array = arr_dict["Residue-Residue Attention"].copy()
cls_flat = cls_array.flatten()
attn_flat = attn_array.flatten()
corr, p_value = pearsonr(cls_flat, attn_flat)

print(f"\nPearson Correlation of [CLS]-to-Residue and Residue-Residue Attention:\n{corr:.1f} (P-value: {p_value})")
print("-> Attention rollout shows that all information flow in the model eventually converges into a single, global 'importance' vector, just like [CLS]'s attention.")

Mean pairwise pearson correlation of residue-residue attention score
mean=0.999999, std=0.0000
-> All residues in a sequence give attention to the same set of sequence in the same degree

[CLS]-to-Residue Attention
Correlation Matrix:
 [[1.   0.68 0.65 0.63 0.62]
 [0.68 1.   0.69 0.65 0.61]
 [0.65 0.69 1.   0.62 0.68]
 [0.63 0.65 0.62 1.   0.71]
 [0.62 0.61 0.68 0.71 1.  ]]
Mean off-diag corr: 0.65

Residue-Residue Attention
Correlation Matrix:
 [[1.   0.68 0.65 0.63 0.62]
 [0.68 1.   0.69 0.65 0.61]
 [0.65 0.69 1.   0.62 0.68]
 [0.63 0.65 0.62 1.   0.71]
 [0.62 0.61 0.68 0.71 1.  ]]
Mean off-diag corr: 0.65

Pearson Correlation of [CLS]-to-Residue and Residue-Residue Attention:
1.0 (P-value: 0.0)
-> Attention rollout shows that all information flow in the model eventually converges into a single, global 'importance' vector, just like [CLS]'s attention.


In [6]:
arr_dict = {
    "Class 1 Attribution": all_attr_array_1,
    "Class 0 Attribution": all_attr_array_0,
}

attr_1_array = arr_dict["Class 1 Attribution"].copy()
attr_0_array = arr_dict["Class 0 Attribution"].copy()
attr_1_flat = attr_1_array.flatten()
attr_0_flat = attr_0_array.flatten()
corr, p_value = pearsonr(attr_1_flat, attr_0_flat)
print(f"Pearson Correlation of Class 1 and Class 0 Attributions:\n{corr:.4f} (P-value: {p_value})")

for name, arr in arr_dict.items():
    print(f"\n{name}")
    reshaped_array = arr.reshape(N_MODEL, -1)
    corr, pvalue = spearmanr(reshaped_array, axis=1)
    #corr = np.corrcoef(reshaped_array)
    print("Correlation Matrix:\n", np.round(corr, 2))
    print("Mean off-diag corr:", np.mean(corr[np.triu_indices_from(corr, k=1)]).round(2))

Pearson Correlation of Class 1 and Class 0 Attributions:
-0.9908 (P-value: 0.0)

Class 1 Attribution
Correlation Matrix:
 [[1.   0.39 0.51 0.53 0.55]
 [0.39 1.   0.46 0.42 0.42]
 [0.51 0.46 1.   0.48 0.5 ]
 [0.53 0.42 0.48 1.   0.49]
 [0.55 0.42 0.5  0.49 1.  ]]
Mean off-diag corr: 0.47

Class 0 Attribution
Correlation Matrix:
 [[1.   0.41 0.5  0.54 0.56]
 [0.41 1.   0.46 0.42 0.42]
 [0.5  0.46 1.   0.47 0.51]
 [0.54 0.42 0.47 1.   0.5 ]
 [0.56 0.42 0.51 0.5  1.  ]]
Mean off-diag corr: 0.48


In [7]:
pos_conflict_mask = (attr_0_array > 0) & (attr_1_array > 0)
neg_conflict_mask = (attr_0_array < 0) & (attr_1_array < 0)
conflict_mask = (pos_conflict_mask | neg_conflict_mask)
align_mask = ~conflict_mask

align_attr_0 = attr_0_array[align_mask]
align_attr_1 = attr_1_array[align_mask]
conflict_attr_0 = attr_0_array[conflict_mask]
conflict_attr_1 = attr_1_array[conflict_mask]

# pos_conflict_attr_0 = attr_0_array[pos_conflict_mask]
# pos_conflict_attr_1 = attr_1_array[pos_conflict_mask]
# neg_conflict_attr_0 = attr_0_array[neg_conflict_mask]
# neg_conflict_attr_1 = attr_1_array[neg_conflict_mask]

data_groups = [
    align_attr_0, align_attr_1,
    conflict_attr_0, conflict_attr_1,
]
titles = [
    'Align (Class 0)', 'Align (Class 1)',
    'Conflict (Class 0)', 'Conflict (Class 1)'
]

fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True)
axes = axes.flatten()
for i, ax in enumerate(axes):
    data = data_groups[i]
    title = titles[i]
    mean_val = np.mean(np.abs(data))
    std_val = np.std(np.abs(data))
    label_text = (
        f"Count: {len(data)}\n"
        f"avg magnitude: {mean_val:.2f}\n"
        f"sd: {std_val:.2f}"
    )
    print(title)
    print(label_text)
    print("------")
    ax.hist(data, bins='auto', alpha=0.75, label=label_text)
    ax.set_title(title)
    ax.set_ylabel('Frequency')
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
axes[2].set_xlabel('Attribution Score')
axes[3].set_xlabel('Attribution Score')
fig.suptitle('Histograms of Attribution Scores', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig("attribution_histograms.png")

Align (Class 0)
Count: 864339
avg magnitude: 0.10
sd: 0.14
------
Align (Class 1)
Count: 864339
avg magnitude: 0.10
sd: 0.14
------
Conflict (Class 0)
Count: 30181
avg magnitude: 0.01
sd: 0.02
------
Conflict (Class 1)
Count: 30181
avg magnitude: 0.01
sd: 0.02
------


In [8]:
abs_attr_score_array = (np.abs(attr_0_array) + np.abs(attr_1_array)) / 2
abs_attr_score_array[conflict_mask] = 0
attr_score_array = abs_attr_score_array.copy()
# assign signs by class 1 attribution
attr_score_array[(align_mask & (attr_1_array < 0))] *= -1

reshaped_array = attr_score_array.reshape(N_MODEL, -1)
corr, pvalue = spearmanr(reshaped_array, axis=1)
#corr = np.corrcoef(reshaped_array)
print("Combined Attribution")
print("Correlation Matrix:\n", np.round(corr, 2))
print("Mean off-diag corr:", np.mean(corr[np.triu_indices_from(corr, k=1)]).round(2))

Combined Attribution
Correlation Matrix:
 [[1.   0.4  0.51 0.54 0.56]
 [0.4  1.   0.46 0.42 0.42]
 [0.51 0.46 1.   0.48 0.5 ]
 [0.54 0.42 0.48 1.   0.49]
 [0.56 0.42 0.5  0.49 1.  ]]
Mean off-diag corr: 0.48


In [9]:
x_data = np.abs(attr_score_array).flatten()
y_data = cls_attn_scalar_array.flatten() #attn_score_array.flatten()
corr, p_value = spearmanr(x_data, y_data)

print(f"Ranking Correlation of Attribution and Attention: {corr:.4f} ({p_value})")
print("-> Residues with high attribution do not always receive high attention.")

plt.figure(figsize=(10, 8))

ax = sns.regplot(
    x=x_data[:10000],
    y=y_data[:10000],
    scatter_kws={'alpha': 0.1, 's': 10}, # Make points small and transparent
    line_kws={'color': 'red', 'linewidth': 2} # Make line red
)

ax.text(0.75, 0.9, f'Spearman r = {corr:.3f}\np = {p_value:.2e}', 
        transform=ax.transAxes, 
        #ha='left', va='top', 
        bbox=dict(boxstyle='round,pad=0.5', fc='white', alpha=0.8))

ax.set_title("Correlation of Attribution and Attention", fontsize=16, fontweight='bold')
ax.set_xlabel("|IG Attribution Score|")
ax.set_ylabel("[CLS]-Residue Attention Score")
plt.savefig("cls_attr_corr.png")

Ranking Correlation of Attribution and Attention: 0.1441 (0.0)
-> Residues with high attribution do not always receive high attention.


In [10]:
vrc01_footprint = set([97, 123, 124, 198, 276, 278, 279, 280, 281, 282, 365, 366, 367, 368, 371, 427, 428, 429, 430, 455, 456, 457, 458, 459, 460, 461, 463, 465, 466, 467, 469, 472, 473, 474, 476])
vrc01_covary = set([46, 132, 138, 144, 150, 179, 181, 186, 190, 290, 321, 328, 354, 389, 394, 396, 397, 406])
vrc01_pngs = set([156, 157, 158, 229, 230, 231, 234, 235, 236, 616, 617, 618, 824, 825, 826])

vrc01_contact_residues = list(vrc01_footprint | vrc01_covary | vrc01_pngs)

In [11]:
vrc01_contact_residues = [str(_) for _ in vrc01_contact_residues]
mask_vrc01_contact_residues = np.isin(resno_array, vrc01_contact_residues)

epitopes = resno_df["Epitope"].unique()
epitope_dfs = resno_df.groupby(by="Epitope")

arr_dict = {
    "Attribution Magnitude": np.abs(np.mean(attr_score_array, axis=0)),
    "[CLS]-to-Residue Attention": np.mean(cls_attn_scalar_array, axis=0),
    "Residue-to-Residue Attention": np.mean(attn_score_array, axis=0),
}

for name, mean_arr in arr_dict.items():
    all_samples = {}
    
    vrc01_samples = (np.sum(mean_arr[:,mask_vrc01_contact_residues], axis=(-1)) / mask_vrc01_contact_residues.sum()).flatten()
    all_samples['VRC01 Contact'] = vrc01_samples

    non_vrc01_samples = (np.sum(mean_arr[:,~mask_vrc01_contact_residues], axis=(-1)) / (~mask_vrc01_contact_residues).sum()).flatten()
    all_samples['Non-VRC01 Contact'] = non_vrc01_samples
    
    for epitope in epitopes:
        epitope_reslabel = epitope_dfs.get_group(epitope)["ResLabel"].unique()
        mask_epitope_respos = np.isin(resno_array, epitope_reslabel)
        if mask_epitope_respos.sum() == 0:
            continue
        epitope_samples = (np.sum(mean_arr[:,mask_epitope_respos], axis=(-1)) / mask_epitope_respos.sum()).flatten()
        group_name = f'Epitope: {epitope.replace("_","-")}'
        all_samples[group_name] = epitope_samples
    
    stats_dict = {}
    for group_name, data in all_samples.items():
        stats_dict[group_name] = {
        'mean': np.mean(data),
        'sd': np.std(data)
        }
    stats_df = pd.DataFrame.from_dict(stats_dict, orient='index')

    group_names = list(all_samples.keys())
    num_groups = len(group_names)
    p_value_matrix = pd.DataFrame(np.nan, index=group_names, columns=group_names)
    annot_matrix = pd.DataFrame("", index=group_names, columns=group_names)
    for group_1, group_2 in itertools.combinations(group_names, 2):
        samples_1 = all_samples[group_1]
        samples_2 = all_samples[group_2]
        _, p_value = ttest_ind(samples_1, samples_2, equal_var=False)
        p_value_matrix.loc[group_1, group_2] = p_value
        p_value_matrix.loc[group_2, group_1] = p_value
        mark = get_significance_mark(p_value, 0.05)
        annot_matrix.loc[group_1, group_2] = mark
        annot_matrix.loc[group_2, group_1] = mark
    
    print(f"--- {name} ---")
    print(stats_df.to_string())
    print(annot_matrix.to_string())

--- Attribution Magnitude ---
                               mean        sd
Epitope: CD4-binding-site  0.074921  0.018164
Epitope: V2-apex           0.098170  0.024150
Epitope: V3-glycan         0.079932  0.020210
Epitope: interface         0.077797  0.016786
Non-VRC01 Contact          0.075037  0.015127
VRC01 Contact              0.097007  0.022952
                          VRC01 Contact Non-VRC01 Contact Epitope: interface Epitope: CD4-binding-site Epitope: V2-apex Epitope: V3-glycan
VRC01 Contact                                         ***                ***                       ***             n.s.                ***
Non-VRC01 Contact                   ***                                  ***                      n.s.              ***                ***
Epitope: interface                  ***               ***                                          ***              ***                  *
Epitope: CD4-binding-site           ***              n.s.                ***                

In [11]:
attr_mean_scores = np.mean(attr_score_array, axis=0)
attr_pred = attr_mean_scores.sum(axis=1)
attr_pred[attr_pred < 0] = 0
attr_pred[attr_pred > 0] = 1
print("confusion matrix for class labels predicted by sum of residue attribution")
print(confusion_matrix(attr_pred, data_df["Label"].values))

confusion matrix for class labels predicted by sum of residue attribution
[[566   3]
 [  3 284]]


In [12]:
def build_typeaware_arrays(sequences, array_scalar):
    """Expand attention and attribution arrays into amino acid-specific channels."""
    N_SEQ, L_SEQ = sequences.shape
    
    aa_idx_array = np.vectorize(aa_to_idx.get, otypes=[int])(sequences)
    array = np.zeros((N_SEQ, L_SEQ, N_AA))
    seq_idx = np.arange(N_SEQ)[:, None]
    pos_idx = np.arange(L_SEQ)[None, :]
    array[seq_idx, pos_idx, aa_idx_array] = array_scalar

    return array

def get_aa_counts(selected_aligned_seq):
    AA_LIST = list("ACDEFGHIKLMNPQRSTVWYO-")
    N_AA = len(AA_LIST)
    aa_to_idx = {aa: i for i, aa in enumerate(AA_LIST)}
    idx_to_aa = {v: k for k, v in aa_to_idx.items()}
    aa_idx_array = np.vectorize(aa_to_idx.get, otypes=[int])(selected_aligned_seq)
    alignments = np.zeros([aa_idx_array.shape[1], N_AA])
    for i in range(aa_idx_array.shape[1]):
        unique, counts = np.unique(aa_idx_array[:,i], return_counts=True)
        alignments[i, unique] = counts
    return alignments

def get_res_significance(scores, aligned_sequences, resno_array=resno_array, idx_to_aa=idx_to_aa):
    type_aware_scores = build_typeaware_arrays(aligned_sequences, scores)
    result_list = []
    for res_idx in range(L_SEQ):
        res_scores = type_aware_scores[:,res_idx]
        for aa_idx_1 in range(N_AA):
            res_aa_score = res_scores[:, aa_idx_1]
            score_1 = res_aa_score[res_aa_score != 0]
            if len(score_1) == 0:
                continue
            
            other_aa_indices = [k for k in idx_to_aa.keys() if k != aa_idx_1]
            not_res_aa_score_2d = res_scores[:, other_aa_indices]
            score_2 = not_res_aa_score_2d[not_res_aa_score_2d != 0].flatten()
            add_zeros = len(set(np.where((not_res_aa_score_2d == 0).all(axis=1))[0]) - set(np.where(res_aa_score != 0)[0]))
            if add_zeros > 0:
                score_2 = np.concatenate([score_2, np.zeros([add_zeros])]).flatten()

            assert len(score_1) + len(score_2) == len(aligned_sequences)

            if len(score_2) == 0:
                result_list.append({
                    "ResLabel": resno_array[res_idx],
                    "AA": idx_to_aa[aa_idx_1],
                    "AA_count": len(score_1),
                    "AA_mean_score": np.mean(score_1),
                    "AA_std_score": np.std(score_1),
                    "nonAA_count": 0,
                    "notAA_mean_score": 0,
                    "notAA_std_score": 0,
                    "pvalue": 0,
                })
                continue
            if len(score_1) > 1 and len(score_2) > 1:
                _, p_value = ttest_ind(score_1, score_2, equal_var=False)
            else:
                p_value = np.nan
            result_list.append({
                "ResLabel": resno_array[res_idx],
                "AA": idx_to_aa[aa_idx_1],
                "AA_count": len(score_1),
                "AA_mean_score": np.mean(score_1),
                "AA_std_score": np.std(score_1),
                "nonAA_count": len(score_2),
                "notAA_mean_score": np.mean(score_2),
                "notAA_std_score": np.std(score_2),
                "pvalue": p_value,
            })
    return pd.DataFrame(result_list)

vrc01_footprint = set([97, 123, 124, 198, 276, 278, 279, 280, 281, 282, 365, 366, 367, 368, 371, 427, 428, 429, 430, 455, 456, 457, 458, 459, 460, 461, 463, 465, 466, 467, 469, 472, 473, 474, 476])
vrc01_covary = set([46, 132, 138, 144, 150, 179, 181, 186, 190, 290, 321, 328, 354, 389, 394, 396, 397, 406])
vrc01_pngs = set([156, 157, 158, 229, 230, 231, 234, 235, 236, 616, 617, 618, 824, 825, 826])

vrc01_contact_residues = list(vrc01_footprint | vrc01_covary | vrc01_pngs)
vrc01_contact_residues = [str(_) for _ in vrc01_contact_residues]

In [13]:
class_0_mask = data_df['Label'] == 0
class_1_mask = data_df['Label'] == 1

class_0_aa_count = get_aa_counts(aligned_sequences[class_0_mask])
class_1_aa_count = get_aa_counts(aligned_sequences[class_1_mask])

aa_df = {"ResLabel":[], "AA":[], "num_class_0":[], "num_class_1":[]}
for i in range(L_SEQ):
    reslabel = resno_array[i]
    for j in range(N_AA):
        aa = idx_to_aa[j]
        class_0_count = int(class_0_aa_count[i][j])
        class_1_count = int(class_1_aa_count[i][j])
        aa_df["ResLabel"].append(reslabel)
        aa_df["AA"].append(aa)
        aa_df["num_class_0"].append(class_0_count)
        aa_df["num_class_1"].append(class_1_count)
aa_df = pd.DataFrame(aa_df)

# Run a two-sample t-test (AA vs. non-AA) for every possible amino acid substitution at every position.
# This identifies statistically significant (p < 0.05) differences in the attention paid 
# to a specific residue/AA combination compared to all others at that site.

print("--- [CLS]-residue attention ---")
cls_mean_scores = np.mean(cls_attn_scalar_array, axis=0)
cls_sig_df = get_res_significance(cls_mean_scores, aligned_sequences)
print(f"The total number of unique AA pairs: {len(cls_sig_df)}")
cls_sig_df = cls_sig_df.merge(resno_df, on=["ResLabel"])
# Mark known VRC01 contact site
cls_sig_df.loc[cls_sig_df["ResLabel"].isin(vrc01_contact_residues), "VRC01_related"] = True
# Filter to select only features that meet the criteria:
# 1) Statistically significant (p < 0.05)
# 2) The specific AA substitution is associated with a higher mean CLS attention score than the non-AA group.
selected_cls_sig_df = cls_sig_df.loc[(cls_sig_df["pvalue"] < 0.05) & 
                                     (cls_sig_df["AA_mean_score"] > cls_sig_df["notAA_mean_score"])
                                    ]
print(f"Number of selected features: {len(selected_cls_sig_df)}")
print(selected_cls_sig_df["Epitope"].value_counts())
vrc01_cls_sig_df = selected_cls_sig_df[selected_cls_sig_df["VRC01_related"] == True]
print(f"Number of features that are specifically VRC01-related: {len(vrc01_cls_sig_df)}")

print("\n--- Attribution ---")
attr_mean_scores = np.mean(attr_score_array, axis=0)
attr_sig_df = get_res_significance(attr_mean_scores, aligned_sequences)
print(f"The total number of unique AA pairs: {len(attr_sig_df)}")
attr_sig_df = attr_sig_df.merge(resno_df, on=["ResLabel"])
# Mark known VRC01 contact site
attr_sig_df.loc[attr_sig_df["ResLabel"].isin(vrc01_contact_residues), "VRC01_related"] = True
# Filter to select only features that meet the criteria:
# 1) Statistically significant (p < 0.05)
# 2) The specific AA substitution is associated with a higher mean CLS attention score than the non-AA group.
selected_attr_sig_df = attr_sig_df.loc[(attr_sig_df["pvalue"] < 0.05) & 
                                       (np.abs(attr_sig_df["AA_mean_score"]) > np.abs(attr_sig_df["notAA_mean_score"]))
                                    ]
print(f"Number of selected features: {len(selected_attr_sig_df)}")
print(selected_attr_sig_df["Epitope"].value_counts())
vrc01_attr_sig_df = selected_attr_sig_df[selected_attr_sig_df["VRC01_related"] == True]
print(f"Number of features that are specifically VRC01-related: {len(vrc01_attr_sig_df)}")

merge_cls_attr = pd.merge(selected_cls_sig_df, selected_attr_sig_df, on=["ResLabel", "AA", "VRC01_related"], suffixes=("_cls", "_attr"), how="inner")
print(f"\nNumber of features that statistically significant by both methods: {len(merge_cls_attr)}")
print(f"Number of VRC01-related features agree in both method: {len(pd.merge(vrc01_cls_sig_df, vrc01_attr_sig_df, on=['ResLabel', 'AA']))}")

# Merges and save the data columns from the significant Attribution features with the statistical summary from the CLS Attention
df = cls_sig_df[["ResLabel", "AA", "AA_mean_score", "AA_std_score", "notAA_mean_score", "notAA_std_score", "pvalue"]]
attr_with_attn = selected_attr_sig_df.merge(df, on=["ResLabel", "AA"], suffixes=("_attr", "_attn"))
assert len(attr_with_attn) == len(selected_attr_sig_df)
attr_with_attn = attr_with_attn.merge(aa_df, on=["ResLabel", "AA"])
assert (attr_with_attn["AA_count"] == attr_with_attn["num_class_0"] + attr_with_attn["num_class_1"]).all()
attr_with_attn.to_csv(os.path.join(root_dir, data_dir, "sig_attr_with_attn.csv"), index=False)

# Isolate and save the features that were significant via Attribution AND located at known VRC01 contact sites.
vrc01_attr_with_attn = attr_with_attn[attr_with_attn["VRC01_related"] == True]
assert len(vrc01_attr_with_attn) == len(vrc01_attr_sig_df)
vrc01_attr_with_attn.to_csv(os.path.join(root_dir, data_dir, "vrc01_sig_attr_with_attn.csv"), index=False)

--- [CLS]-residue attention ---
The total number of unique AA pairs: 1672
Number of selected features: 399
CD4_binding_site    189
interface            84
V3_glycan            72
V2_apex              54
Name: Epitope, dtype: int64
Number of features that are specifically VRC01-related: 73

--- Attribution ---
The total number of unique AA pairs: 1672
Number of selected features: 531
CD4_binding_site    214
interface           131
V3_glycan           103
V2_apex              83
Name: Epitope, dtype: int64
Number of features that are specifically VRC01-related: 102

Number of features that statistically significant by both methods: 198
Number of VRC01-related features agree in both method: 33
